# 02 — Préparation des données

**Source unique : `data/JO_resultats.csv`** (fourni dans le repo, equipe projet)
- Couvre 1896→2024 (toutes éditions, été + hiver), separateur `;`
- Match exact CIO sur 2016, 2020, 2024 (validation effectuée)
- 1 ligne = 1 athlète médaillé (relais => N lignes par médaille => déduplication nécessaire)
- `data/JO2028_sports.csv` : programme officiel des disciplines de LA 2028 (pour reperer les nouveaux sports)

## Étapes
1. Charge `JO_resultats.csv` + `JO2028_sports.csv`
2. Filtre JO d'été uniquement
3. Déduplique les épreuves par équipe (relais, sports collectifs)
4. Normalise les noms de sports
5. Agrège par (Year, NOC, Sport) → Gold/Silver/Bronze/Total
6. Enrichit avec Population + Continent
7. Exporte `medals_per_country_sport.csv`

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if (Path.cwd().name == "notebooks") else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

RESULTS_CSV = DATA_DIR / "JO_resultats.csv"
SPORTS_2028_CSV = DATA_DIR / "JO2028_sports.csv"
print(f"Lecture resultats : {RESULTS_CSV}")
print(f"Lecture programme 2028 : {SPORTS_2028_CSV}")

Lecture resultats : /Users/nicolaschalopin/Documents/github/jo-2028-predictions/main/data/JO_resultats.csv
Lecture programme 2028 : /Users/nicolaschalopin/Documents/github/jo-2028-predictions/main/data/JO2028_sports.csv


## 1. Chargement

In [2]:
# JO_resultats.csv utilise le separateur ';'
oly = pd.read_csv(RESULTS_CSV, sep=";", low_memory=False)
print(f"Lignes brutes      : {len(oly):,}")
print(f"Années             : {oly['event_year'].nunique()} ({oly['event_year'].min()}→{oly['event_year'].max()})")
print(f"Pays uniques       : {oly['country_code'].nunique()}")
print(f"Disciplines        : {oly['discipline'].nunique()}")
print(f"Distribution medal : {oly['medal_type'].value_counts().to_dict()}")
oly.head(3)

Lignes brutes      : 24,012
Années             : 38 (1896→2024)
Pays uniques       : 160
Disciplines        : 87
Distribution medal : {'Bronze': 8336, 'Gold': 7861, 'Silver': 7815}


,discipline,event,event_gender,event_type,medal_type,name,country_code,country,city,event_year
0,Cycling Road,Men Individual Time Trial,Male,Athlete,Gold,Evenepoel Remco,BEL,Belgium,Paris,2024
1,Cycling Road,Men Individual Time Trial,Male,Athlete,Silver,Ganna Filippo,ITA,Italy,Paris,2024
2,Cycling Road,Men Individual Time Trial,Male,Athlete,Bronze,Van Aert Wout,BEL,Belgium,Paris,2024


## 2. Filtre JO d'été

Le dataset contient été + hiver. On garde uniquement les éditions d'été (sujet du projet).

In [3]:
SUMMER_YEARS = [
    1896, 1900, 1904, 1908, 1912, 1920, 1924, 1928, 1932, 1936,
    1948, 1952, 1956, 1960, 1964, 1968, 1972, 1976, 1980, 1984,
    1988, 1992, 1996, 2000, 2004, 2008, 2012, 2016, 2020, 2024,
]

summer = oly[oly["event_year"].isin(SUMMER_YEARS)].copy()
print(f"Lignes JO d'été : {len(summer):,}")
print(f"Éditions retenues : {sorted(summer['event_year'].unique())}")

Lignes JO d'été : 21,799
Éditions retenues : [np.int64(1896), np.int64(1900), np.int64(1904), np.int64(1908), np.int64(1912), np.int64(1920), np.int64(1924), np.int64(1928), np.int64(1932), np.int64(1936), np.int64(1948), np.int64(1952), np.int64(1956), np.int64(1960), np.int64(1964), np.int64(1968), np.int64(1972), np.int64(1976), np.int64(1980), np.int64(1984), np.int64(1988), np.int64(1992), np.int64(1996), np.int64(2000), np.int64(2004), np.int64(2008), np.int64(2012), np.int64(2016), np.int64(2020), np.int64(2024)]


## 3. Filtrage des médaillés + déduplication

Chaque médaille collective (relais, équipe) est dupliquée par membre. On déduplique
sur (year, discipline, event, medal_type, country_code) pour ne compter qu'une
médaille par épreuve.

In [4]:
medals = summer[summer["medal_type"].isin(["Gold", "Silver", "Bronze"])].copy()
print(f"Lignes avec médaille (brut) : {len(medals):,}")

medals_unique = medals.drop_duplicates(
    subset=["event_year", "discipline", "event", "medal_type", "country_code"]
).copy()
print(f"Médailles uniques (après dedup) : {len(medals_unique):,}")

totals = medals_unique.groupby("event_year").size()
print(f"\nTotal médailles attribuées par édition (dernières) :")
print(totals.tail(8))

Lignes avec médaille (brut) : 21,799
Médailles uniques (après dedup) : 19,128

Total médailles attribuées par édition (dernières) :
event_year
1996     840
2000     923
2004     925
2008     953
2012     957
2016     973
2020    1080
2024    1043
dtype: int64


## 4. Normalisation des noms de sports

On garde les disciplines telles quelles mais on supprime les suffixes parasites
type `(Aquatics)` rencontrés sur d'autres datasets.

In [5]:
medals_unique["Sport"] = (
    medals_unique["discipline"]
    .str.replace(r"\s*\([^)]*\)", "", regex=True)
    .str.strip()
)

print(f"Sports uniques : {medals_unique['Sport'].nunique()}")
print(f"Top 15 sports par volume :")
print(medals_unique["Sport"].value_counts().head(15))

Sports uniques : 85
Top 15 sports par volume :
Sport
Athletics              3220
Swimming               1865
Wrestling              1428
Boxing                 1048
Gymnastics Artistic     962
Shooting                906
Rowing                  849
Weightlifting           709
Fencing                 678
Judo                    665
Canoe Sprint            599
Sailing                 585
Cycling Track           551
Diving                  412
Speed skating           324
Name: count, dtype: int64


## 5. Agrégation par (Year, NOC, Sport)

In [6]:
pivot = medals_unique.pivot_table(
    index=["event_year", "country_code", "country", "Sport"],
    columns="medal_type",
    values="event",
    aggfunc="count",
    fill_value=0,
).reset_index()

pivot.columns.name = None
pivot = pivot.rename(columns={
    "event_year": "Year",
    "country_code": "NOC",
    "country": "Country",
})

for col in ["Gold", "Silver", "Bronze"]:
    if col not in pivot.columns:
        pivot[col] = 0
    pivot[col] = pivot[col].astype(int)

pivot["Total"] = pivot["Gold"] + pivot["Silver"] + pivot["Bronze"]
pivot["Year"] = pivot["Year"].astype(int)

print(f"Lignes agrégées : {len(pivot):,}")
print(f"Pays uniques    : {pivot['NOC'].nunique()}")
print(f"Sports uniques  : {pivot['Sport'].nunique()}")
print(f"Éditions        : {pivot['Year'].nunique()}")
pivot.head()

Lignes agrégées : 7,454
Pays uniques    : 159
Sports uniques  : 85
Éditions        : 30


,Year,NOC,Country,Sport,Bronze,Gold,Silver,Total
0,1896,AUS,Australia,Athletics,0,2,0,2
1,1896,AUT,Austria,Cycling Track,2,1,0,3
2,1896,AUT,Austria,Swimming,0,1,1,2
3,1896,DEN,Denmark,Fencing,1,0,0,1
4,1896,DEN,Denmark,Shooting,2,0,1,3


## 6. Validation vs chiffres officiels CIO

In [7]:
def check(year, noc, expected_g, expected_s, expected_b):
    sub = pivot[(pivot["Year"] == year) & (pivot["NOC"] == noc)]
    g, s, b = sub["Gold"].sum(), sub["Silver"].sum(), sub["Bronze"].sum()
    t = g + s + b
    expected_t = expected_g + expected_s + expected_b
    match = (g == expected_g) and (s == expected_s) and (b == expected_b)
    flag = "OK   " if match else "ECART"
    print(f"  [{flag}] {year} {noc:4s} : "
          f"obtenu {g:2d}-{s:2d}-{b:2d}={t:3d} | CIO {expected_g:2d}-{expected_s:2d}-{expected_b:2d}={expected_t:3d}")
    return match

print("=== VALIDATION CIO ===")
check(2024, "USA", 40, 44, 42)
check(2024, "CHN", 40, 27, 24)
check(2024, "JPN", 20, 12, 13)
check(2024, "AUS", 18, 19, 16)
check(2020, "USA", 39, 41, 33)
check(2020, "CHN", 38, 32, 19)
check(2020, "JPN", 27, 14, 17)
check(2016, "USA", 46, 37, 38)
check(2016, "GBR", 27, 23, 17)
check(2016, "CHN", 26, 18, 26)
check(2012, "USA", 48, 26, 30)
check(2008, "CHN", 48, 22, 30)

=== VALIDATION CIO ===
  [OK   ] 2024 USA  : obtenu 40-44-42=126 | CIO 40-44-42=126
  [OK   ] 2024 CHN  : obtenu 40-27-24= 91 | CIO 40-27-24= 91
  [OK   ] 2024 JPN  : obtenu 20-12-13= 45 | CIO 20-12-13= 45
  [OK   ] 2024 AUS  : obtenu 18-19-16= 53 | CIO 18-19-16= 53
  [OK   ] 2020 USA  : obtenu 39-41-33=113 | CIO 39-41-33=113
  [ECART] 2020 CHN  : obtenu 38-32-18= 88 | CIO 38-32-19= 89
  [OK   ] 2020 JPN  : obtenu 27-14-17= 58 | CIO 27-14-17= 58
  [OK   ] 2016 USA  : obtenu 46-37-38=121 | CIO 46-37-38=121
  [OK   ] 2016 GBR  : obtenu 27-23-17= 67 | CIO 27-23-17= 67
  [OK   ] 2016 CHN  : obtenu 26-18-26= 70 | CIO 26-18-26= 70
  [ECART] 2012 USA  : obtenu 47-27-30=104 | CIO 48-26-30=104
  [OK   ] 2008 CHN  : obtenu 48-22-30=100 | CIO 48-22-30=100


np.True_

## 6bis. Programme officiel LA 2028 + sports nouveaux

`JO2028_sports.csv` liste les disciplines au programme de Los Angeles 2028.
On compare avec les sports vus historiquement pour reperer les **sports nouveaux**
(cricket, lacrosse, flag football, baseball/softball, squash) : sans historique,
le modele ne pourra pas les predire — limite assumee.

In [8]:
sports_2028 = pd.read_csv(SPORTS_2028_CSV, sep=";")
sports_2028["Sport"] = (
    sports_2028["Discipline"]
    .str.replace(r"\s*\([^)]*\)", "", regex=True)
    .str.strip()
)
disciplines_2028 = set(sports_2028["Sport"].unique())
print(f"Disciplines au programme LA 2028 : {len(disciplines_2028)}")
print(f"Epreuves totales LA 2028         : {len(sports_2028)}")

# Sports historiques (presents dans nos donnees agregees)
sports_hist = set(pivot["Sport"].unique())

# Sports 2028 sans aucun historique => non predictibles
nouveaux_2028 = sorted(disciplines_2028 - sports_hist)
print(f"\nSports LA 2028 SANS historique ({len(nouveaux_2028)}) -- non predictibles :")
for s in nouveaux_2028:
    print(f"  - {s}")

Disciplines au programme LA 2028 : 51
Epreuves totales LA 2028         : 352

Sports LA 2028 SANS historique (3) -- non predictibles :
  - Flag Football
  - Rowing Coastal Beach Sprint
  - Squash


## 7. Enrichissement Population + Continent

In [9]:
NOC_TO_CONTINENT = {
    "FRA": "Europe", "GBR": "Europe", "GER": "Europe", "ITA": "Europe", "ESP": "Europe",
    "NED": "Europe", "BEL": "Europe", "POR": "Europe", "SUI": "Europe", "AUT": "Europe",
    "POL": "Europe", "HUN": "Europe", "CZE": "Europe", "GRE": "Europe", "DEN": "Europe",
    "NOR": "Europe", "SWE": "Europe", "FIN": "Europe", "IRL": "Europe", "ROU": "Europe",
    "BUL": "Europe", "CRO": "Europe", "SRB": "Europe", "SVK": "Europe", "SLO": "Europe",
    "UKR": "Europe", "RUS": "Europe", "BLR": "Europe", "GEO": "Europe", "ARM": "Europe",
    "AZE": "Europe", "TUR": "Europe", "EST": "Europe", "LAT": "Europe", "LTU": "Europe",
    "ISL": "Europe", "LUX": "Europe", "MDA": "Europe", "MNE": "Europe", "MKD": "Europe",
    "ALB": "Europe", "BIH": "Europe", "KOS": "Europe", "CYP": "Europe", "MLT": "Europe",
    "AIN": "Europe", "ROC": "Europe", "EUN": "Europe", "URS": "Europe",
    "GDR": "Europe", "FRG": "Europe", "TCH": "Europe", "YUG": "Europe",
    "USA": "Americas", "CAN": "Americas", "MEX": "Americas", "BRA": "Americas",
    "ARG": "Americas", "CHI": "Americas", "COL": "Americas", "PER": "Americas",
    "VEN": "Americas", "ECU": "Americas", "URU": "Americas", "PAR": "Americas",
    "BOL": "Americas", "CUB": "Americas", "DOM": "Americas", "PUR": "Americas",
    "JAM": "Americas", "TTO": "Americas", "BAH": "Americas", "GRN": "Americas",
    "GUA": "Americas", "CRC": "Americas", "PAN": "Americas", "HON": "Americas",
    "ESA": "Americas", "BAR": "Americas", "DMA": "Americas", "SUR": "Americas",
    "HAI": "Americas", "ISV": "Americas",
    "CHN": "Asia", "JPN": "Asia", "KOR": "Asia", "PRK": "Asia", "TPE": "Asia",
    "HKG": "Asia", "MGL": "Asia", "VIE": "Asia", "THA": "Asia", "INA": "Asia",
    "MAS": "Asia", "PHI": "Asia", "SGP": "Asia", "IND": "Asia", "PAK": "Asia",
    "SRI": "Asia", "BAN": "Asia", "NEP": "Asia", "AFG": "Asia", "IRI": "Asia",
    "IRQ": "Asia", "SYR": "Asia", "JOR": "Asia", "LBN": "Asia", "ISR": "Asia",
    "PLE": "Asia", "KSA": "Asia", "UAE": "Asia", "QAT": "Asia", "BRN": "Asia",
    "KUW": "Asia", "OMA": "Asia", "YEM": "Asia", "KAZ": "Asia", "UZB": "Asia",
    "KGZ": "Asia", "TJK": "Asia", "TKM": "Asia", "CAM": "Asia", "LAO": "Asia",
    "MYA": "Asia", "BHU": "Asia", "MDV": "Asia", "TLS": "Asia",
    "RSA": "Africa", "EGY": "Africa", "MAR": "Africa", "TUN": "Africa", "ALG": "Africa",
    "KEN": "Africa", "ETH": "Africa", "NGR": "Africa", "GHA": "Africa", "CIV": "Africa",
    "SEN": "Africa", "CMR": "Africa", "UGA": "Africa", "TAN": "Africa", "ZIM": "Africa",
    "BOT": "Africa", "NAM": "Africa", "MOZ": "Africa", "ANG": "Africa", "ZAM": "Africa",
    "MAD": "Africa", "MLI": "Africa", "BUR": "Africa", "NIG": "Africa", "TOG": "Africa",
    "BEN": "Africa", "GAB": "Africa", "COD": "Africa", "CGO": "Africa", "RWA": "Africa",
    "BDI": "Africa", "ERI": "Africa", "SUD": "Africa", "DJI": "Africa",
    "LBR": "Africa", "SLE": "Africa", "GAM": "Africa", "GBS": "Africa", "GUI": "Africa",
    "CPV": "Africa", "STP": "Africa", "COM": "Africa", "MRI": "Africa", "SEY": "Africa",
    "LES": "Africa", "SWZ": "Africa", "MAW": "Africa", "MTN": "Africa",
    "LBA": "Africa", "CAF": "Africa", "CHA": "Africa", "EQG": "Africa", "BIZ": "Africa",
    "AUS": "Oceania", "NZL": "Oceania", "FIJ": "Oceania", "PNG": "Oceania",
    "SAM": "Oceania", "TGA": "Oceania", "VAN": "Oceania", "SOL": "Oceania",
    "KIR": "Oceania", "NRU": "Oceania", "PLW": "Oceania", "FSM": "Oceania",
    "MHL": "Oceania", "TUV": "Oceania", "COK": "Oceania", "ANZ": "Oceania",
    "IOA": "Other", "EOR": "Other", "ROT": "Other", "BOH": "Europe", "WIF": "Americas",
}

pivot["Continent"] = pivot["NOC"].map(NOC_TO_CONTINENT).fillna("Other")
print(f"Continents : {pivot['Continent'].value_counts().to_dict()}")
unknown = pivot[pivot["Continent"] == "Other"]["NOC"].unique()
print(f"NOCs sans continent ({len(unknown)}): {sorted(unknown)[:20]}")

Continents : {'Europe': 4774, 'Americas': 1213, 'Asia': 889, 'Oceania': 313, 'Africa': 226, 'Other': 39}
NOCs sans continent (11): ['AHO', 'BER', 'EOR', 'GUY', 'IOA', 'LCA', 'LIE', 'MIX', 'SCG', 'SMR', 'UAR']


In [10]:
POPULATION_M = {
    "USA": 331, "CHN": 1402, "IND": 1380, "RUS": 144, "JPN": 126,
    "GER": 83, "FRA": 67, "GBR": 67, "ITA": 60, "BRA": 213,
    "KOR": 52, "ESP": 47, "AUS": 26, "CAN": 38, "NED": 17,
    "POL": 38, "TUR": 84, "UKR": 44, "EGY": 102, "RSA": 60,
    "IRI": 84, "THA": 70, "ARG": 45, "MEX": 129, "INA": 274,
    "VIE": 97, "PHI": 110, "ETH": 117, "KEN": 54, "NGR": 206,
    "COL": 51, "AFG": 39, "ALG": 44, "MAR": 37, "PER": 33,
    "VEN": 28, "MGL": 3, "GRE": 11, "POR": 10, "BEL": 12,
    "CZE": 11, "HUN": 10, "AUT": 9, "SUI": 9, "SWE": 10,
    "SRB": 7, "NOR": 5, "DEN": 6, "FIN": 6, "IRL": 5,
    "NZL": 5, "JAM": 3, "CUB": 11, "GEO": 4, "ARM": 3,
    "AZE": 10, "BUL": 7, "CRO": 4, "SVK": 5, "SLO": 2,
    "EST": 1, "LAT": 2, "LTU": 3, "LUX": 1, "MLT": 1,
    "ISL": 0.4, "BLR": 9, "KAZ": 19, "UZB": 34, "TPE": 24,
    "HKG": 7, "SGP": 6, "QAT": 3, "UAE": 10, "KSA": 35,
    "BRN": 1.5, "JOR": 10, "LBN": 7, "ISR": 9, "SYR": 17,
    "PRK": 26, "GHA": 31, "CIV": 26, "SEN": 17, "CMR": 27,
    "UGA": 46, "ZIM": 15, "BOT": 2, "NAM": 3, "TUN": 12,
    "MOZ": 31, "ZAM": 18, "TAN": 60, "MAD": 28, "DOM": 11,
    "TTO": 1.4, "BAR": 0.3, "PAN": 4, "GUA": 17, "ECU": 18,
    "URU": 3, "CHI": 19, "FIJ": 0.9, "PNG": 9, "SAM": 0.2,
    "BAH": 0.4, "ALB": 3, "MNE": 0.6, "MKD": 2, "MDA": 3,
    "PUR": 3, "BAN": 165, "PAK": 220, "SRI": 22, "TKM": 6,
    "TJK": 9, "KGZ": 6, "MAS": 32, "ROU": 19, "KOS": 2,
    "NIG": 24, "MTN": 4, "LBA": 7, "GAB": 2,
    "DJI": 1, "AIN": 144, "ROC": 144, "EUN": 290,
    "URS": 290, "GDR": 17, "FRG": 60, "TCH": 15, "YUG": 22,
}

pivot["Population"] = pivot["NOC"].map(POPULATION_M).fillna(5) * 1_000_000
pivot["Population"] = pivot["Population"].astype(int)
print(f"Population renseignée : {(pivot['Population'] > 5_000_000).mean()*100:.1f}% des lignes")

Population renseignée : 90.7% des lignes


## 8. Export final

In [11]:
final = pivot[["Year", "NOC", "Country", "Sport", "Gold", "Silver", "Bronze", "Total",
               "Population", "Continent"]].copy()
final = final.sort_values(["Year", "NOC", "Sport"]).reset_index(drop=True)

assert ((final["Gold"] + final["Silver"] + final["Bronze"]) == final["Total"]).all()
assert final[["Gold", "Silver", "Bronze"]].min().min() >= 0
assert final.duplicated(subset=["Year", "NOC", "Sport"]).sum() == 0

out_path = PROCESSED_DIR / "medals_per_country_sport.csv"
final.to_csv(out_path, index=False)
print(f"Sauvegardé : {out_path}")
print(f"Lignes     : {len(final):,}")
print(f"Taille     : {out_path.stat().st_size / 1024:.1f} Ko")
print(f"\nAperçu :")
final.head(10)

Sauvegardé : /Users/nicolaschalopin/Documents/github/jo-2028-predictions/main/data/processed/medals_per_country_sport.csv
Lignes     : 7,454
Taille     : 407.1 Ko

Aperçu :


,Year,NOC,Country,Sport,Gold,Silver,Bronze,Total,Population,Continent
0,1896,AUS,Australia,Athletics,2,0,0,2,26000000,Oceania
1,1896,AUT,Austria,Cycling Track,1,0,2,3,9000000,Europe
2,1896,AUT,Austria,Swimming,1,1,0,2,9000000,Europe
3,1896,DEN,Denmark,Fencing,0,0,1,1,6000000,Europe
4,1896,DEN,Denmark,Shooting,0,1,2,3,6000000,Europe
5,1896,DEN,Denmark,Weightlifting,1,1,0,2,6000000,Europe
6,1896,FRA,France,Athletics,0,1,1,2,67000000,Europe
7,1896,FRA,France,Cycling Track,4,1,1,6,67000000,Europe
8,1896,FRA,France,Fencing,1,2,0,3,67000000,Europe
9,1896,GBR,Great Britain,Athletics,0,1,1,2,67000000,Europe
